# 🛡️ Day 3 — NumPy & Data Preprocessing

### CyberShield AI — Member 1

Objective: learn NumPy basics and prepare the CICIDS2017 DDoS dataset for Machine Learning.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd

## 2. NumPy Basics

In [2]:
arr = np.array([10, 20, 30, 40, 50])
print(arr)
print(arr.shape)
print(arr.dtype)

[10 20 30 40 50]
(5,)
int64


In [3]:
print("Mean:", np.mean(arr))
print("Median:", np.median(arr))
print("Minimum:", np.min(arr))
print("Maximum:", np.max(arr))

Mean: 30.0
Median: 30.0
Minimum: 10
Maximum: 50


In [4]:
arr2 = np.array([10, 20, np.nan, 40, 50])
print(arr2)
print('NaN values:', np.isnan(arr2))

[10. 20. nan 40. 50.]
NaN values: [False False  True False False]


In [5]:
arr3 = np.array([10, 20, np.inf, 40, -np.inf])
print(arr3)
print('Infinite values:', np.isinf(arr3))

[ 10.  20.  inf  40. -inf]
Infinite values: [False False  True False  True]


## 3. Load CICIDS2017

In [6]:
attack_df = pd.read_csv("../data/CICIDS2017/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
print("Dataset shape:", attack_df.shape)

Dataset shape: (225745, 79)


In [7]:
attack_df.columns = attack_df.columns.str.strip()
print(attack_df.columns.tolist()[:10])

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std']


## 4. NumPy on Real Data

In [8]:
print("Mean Flow Duration:", np.mean(attack_df["Flow Duration"]))
print("Median Flow Duration:", np.median(attack_df["Flow Duration"]))
print("Minimum Flow Duration:", np.min(attack_df["Flow Duration"]))
print("Maximum Flow Duration:", np.max(attack_df["Flow Duration"]))

Mean Flow Duration: 16241648.528131299
Median Flow Duration: 1452333.0
Minimum Flow Duration: -1
Maximum Flow Duration: 119999937


## 5. Data Quality — NaN and Infinity

In [9]:
numeric = attack_df.select_dtypes(include=np.number)
print("NaN values:", np.isnan(numeric).sum().sum())
print("Infinite values:", np.isinf(numeric).sum().sum())

NaN values: 4
Infinite values: 64


In [10]:
nan_counts = np.isnan(attack_df.select_dtypes(include=np.number)).sum()
print("NaN by column:")
print(nan_counts[nan_counts > 0])

inf_counts = np.isinf(attack_df.select_dtypes(include=np.number)).sum()
print("\nInfinity by column:")
print(inf_counts[inf_counts > 0])

NaN by column:
Flow Bytes/s    4
dtype: int64

Infinity by column:
Flow Bytes/s      30
Flow Packets/s    34
dtype: int64


## 6. Investigate Zero-Duration Flows

In [11]:
problem_rows = attack_df[
    attack_df["Flow Bytes/s"].isna() |
    np.isinf(attack_df["Flow Bytes/s"]) |
    np.isinf(attack_df["Flow Packets/s"])
]
print(problem_rows[["Flow Duration","Total Fwd Packets","Total Backward Packets","Flow Bytes/s","Flow Packets/s","Label"]])

        Flow Duration  Total Fwd Packets  Total Backward Packets  \
65                  0                  2                       0   
1767                0                  2                       0   
1890                0                  1                       1   
3375                0                  1                       1   
6796                0                  2                       0   
8057                0                  1                       1   
8405                0                  2                       0   
13313               0                  2                       0   
13716               0                  2                       0   
14739               0                  1                       1   
15047               0                  2                       0   
18253               0                  2                       0   
33330               0                  2                       0   
55551               0                  1        

In [12]:
print("Zero-duration flows:", (attack_df["Flow Duration"] == 0).sum())
print(attack_df[attack_df["Flow Duration"] == 0]["Label"].value_counts())

Zero-duration flows: 34
Label
BENIGN    32
DDoS       2
Name: count, dtype: int64


## 7. Clean a Working Copy

In [13]:
clean_df = attack_df.copy()
clean_df = clean_df.replace([np.inf, -np.inf], np.nan)
print('NaN after converting infinity:', clean_df.isnull().sum().sum())

NaN after converting infinity: 68


In [14]:
nan_counts_clean = clean_df.isnull().sum()
print(nan_counts_clean[nan_counts_clean > 0])

Flow Bytes/s      34
Flow Packets/s    34
dtype: int64


In [15]:
clean_df["Flow Bytes/s"] = clean_df["Flow Bytes/s"].fillna(clean_df["Flow Bytes/s"].median())
clean_df["Flow Packets/s"] = clean_df["Flow Packets/s"].fillna(clean_df["Flow Packets/s"].median())
print("Remaining NaN values:", clean_df.isnull().sum().sum())

Remaining NaN values: 0


## 8. Verify Cleaning

In [16]:
print("Original shape:", attack_df.shape)
print("Cleaned shape:", clean_df.shape)
print("Remaining infinite values:", np.isinf(clean_df.select_dtypes(include=np.number)).sum().sum())

Original shape: (225745, 79)
Cleaned shape: (225745, 79)
Remaining infinite values: 0


## 9. Data Types

In [17]:
print(clean_df.dtypes.value_counts())
print("Non-numerical columns:", clean_df.select_dtypes(exclude=np.number).columns.tolist())

int64      54
float64    24
str         1
Name: count, dtype: int64
Non-numerical columns: ['Label']


## 10. Features and Target

In [18]:
X = clean_df.drop("Label", axis=1)
y = clean_df["Label"]
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (225745, 78)
Target shape: (225745,)


## 11. Label Distribution

In [19]:
print(y.value_counts())
print("\nPercentage distribution:")
print(y.value_counts(normalize=True) * 100)

Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

Percentage distribution:
Label
DDoS      56.713105
BENIGN    43.286895
Name: proportion, dtype: float64


## 12. Feature Statistics

In [20]:
X.describe().T

,count,mean,std,min,25%,50%,75%,max
Destination Port,225745.0,8.879619e+03,1.975465e+04,0.0,80.0,80.0,80.0,65532.0
Flow Duration,225745.0,1.624165e+07,3.152437e+07,-1.0,71180.0,1452333.0,8805237.0,119999937.0
Total Fwd Packets,225745.0,4.874916e+00,1.542287e+01,1.0,2.0,3.0,5.0,1932.0
Total Backward Packets,225745.0,4.572775e+00,2.175536e+01,0.0,1.0,4.0,5.0,2942.0
Total Length of Fwd Packets,225745.0,9.394633e+02,3.249403e+03,0.0,26.0,30.0,63.0,183012.0
...,...,...,...,...,...,...,...,...
Active Min,225745.0,1.776201e+05,7.842602e+05,0.0,0.0,0.0,1862.0,100000000.0
Idle Mean,225745.0,1.032214e+07,2.185303e+07,0.0,0.0,0.0,8239725.0,120000000.0
Idle Std,225745.0,3.611943e+06,1.275689e+07,0.0,0.0,0.0,0.0,65300000.0
Idle Max,225745.0,1.287813e+07,2.692126e+07,0.0,0.0,0.0,8253838.0,120000000.0


## 13. Investigate `-1` Values

In [21]:
print("Flow Duration = -1:", (clean_df["Flow Duration"] == -1).sum())
print(clean_df[clean_df["Flow Duration"] == -1]["Label"].value_counts())

minus_one_counts = (X == -1).sum()
print("\n-1 by feature:")
print(minus_one_counts[minus_one_counts > 0])

Flow Duration = -1: 2
Label
BENIGN    2
Name: count, dtype: int64

-1 by feature:
Flow Duration                  2
Flow IAT Mean                  2
Flow IAT Max                   2
Flow IAT Min                 104
Fwd IAT Min                    5
Init_Win_bytes_forward     32925
Init_Win_bytes_backward    88299
dtype: int64


In [22]:
for col in ["Flow Duration","Flow IAT Mean","Flow IAT Max","Flow IAT Min","Fwd IAT Min"]:
    print(f"\n{col}")
    print(clean_df[clean_df[col] == -1]["Label"].value_counts())


Flow Duration
Label
BENIGN    2
Name: count, dtype: int64

Flow IAT Mean
Label
BENIGN    2
Name: count, dtype: int64

Flow IAT Max
Label
BENIGN    2
Name: count, dtype: int64

Flow IAT Min
Label
BENIGN    86
DDoS      18
Name: count, dtype: int64

Fwd IAT Min
Label
DDoS    5
Name: count, dtype: int64


In [23]:
print("Init_Win_bytes_forward = -1:")
print(clean_df[clean_df["Init_Win_bytes_forward"] == -1]["Label"].value_counts())
print("\nInit_Win_bytes_backward = -1:")
print(clean_df[clean_df["Init_Win_bytes_backward"] == -1]["Label"].value_counts())

Init_Win_bytes_forward = -1:
Label
BENIGN    32925
Name: count, dtype: int64

Init_Win_bytes_backward = -1:
Label
DDoS      46528
BENIGN    41771
Name: count, dtype: int64


**Decision:** `-1` values were investigated and are not blindly replaced or deleted.

In [24]:
print(X.select_dtypes(exclude=np.number).columns.tolist())

[]


## 14. Compare Feature Scales

In [25]:
print("Flow Duration range:", X["Flow Duration"].min(), "to", X["Flow Duration"].max())
print("Total Fwd Packets range:", X["Total Fwd Packets"].min(), "to", X["Total Fwd Packets"].max())

Flow Duration range: -1 to 119999937
Total Fwd Packets range: 1 to 1932


## 15. Train/Test Split

In [26]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (180596, 78)
X_test : (45149, 78)
y_train: (180596,)
y_test : (45149,)


## 16. Standard Scaling

The scaler is fitted only on training data to avoid data leakage.

In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

Scaled training shape: (180596, 78)
Scaled testing shape: (45149, 78)


### Before and after scaling

In [28]:
print("Before scaling:")
print(X_train["Flow Duration"].head().to_list())
print("\nAfter scaling:")
print(X_train_scaled[:5, 1])

Before scaling:
[100125, 1724442, 926193, 95765, 1610116]

After scaling:
[-0.51240248 -0.46087473 -0.48619736 -0.51254079 -0.46450146]


### Verify scaling

In [29]:
print("Mean of each feature:")
print(np.mean(X_train_scaled, axis=0)[:10])
print("\nStd of each feature:")
print(np.std(X_train_scaled, axis=0)[:10])

Mean of each feature:
[ 1.85705205e-17  9.67870346e-18 -4.09180959e-18 -1.57377292e-17
 -3.30492313e-18  3.77705501e-18 -1.93574069e-17  7.39673273e-18
  5.02033562e-17  4.28066234e-17]

Std of each feature:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [30]:
print("Scaler fitted on training data:")
print(scaler.mean_[:5])
print("Training data shape:", X_train_scaled.shape)
print("Testing data shape:", X_test_scaled.shape)

Scaler fitted on training data:
[8.87916627e+03 1.62526646e+07 4.87398946e+00 4.57006800e+00
 9.37603795e+02]
Training data shape: (180596, 78)
Testing data shape: (45149, 78)


# 🛡️ Day 3 — Key Findings

- CICIDS2017 DDoS dataset: **225,745 rows × 79 columns**
- **78 features** and **1 target (`Label`)**
- Original NaN values: **4**
- Original infinite values: **64**
- Zero-duration flows: **34** → 32 BENIGN, 2 DDoS
- Infinity was converted to NaN and affected NaN values were filled with column medians.
- Final NaN count: **0**
- Final infinite count: **0**
- Original data remained in `attack_df`; working copy is `clean_df`.
- Label distribution: DDoS **128,027 (56.71%)**, BENIGN **97,718 (43.29%)**
- `X` contains 78 numerical features; `y` contains `Label`.
- Train/test split: **80/20**, with `stratify=y`.
- `X_train`: `(180596, 78)`
- `X_test`: `(45149, 78)`
- StandardScaler was fitted only on `X_train` and then used to transform both train and test data.
- Each training feature has mean approximately 0 and standard deviation approximately 1.

**Workflow:** CICIDS2017 → NumPy analysis → data-quality checks → cleaning → X/y → train/test split → scaling → ML-ready features.